# ModAdd p=7, m=127 Eval Curves

This notebook builds per-eta eval plots for the new Hydra-based modular-addition runs:

- `Offline BC MC`
- `NAIL-OPD MC`
- `OPD MC`

The default configuration below targets:

- `p = 7`
- `m = 127`
- `subset_size = 1000000`
- `seed = 20260417`
- `eta in {0.0, 0.1, 0.3, 0.5, 0.7, 0.9}`


In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Resolve the repo root whether the notebook is opened from the repo root or the notebooks/ folder.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.plot_modadd_hydra_runs import (
    build_runs_df,
    default_output_dir,
    discover_modadd_runs,
    plot_per_eta,
)

# Search recursively from the repo root so nested Hydra suite directories are included.
RUN_ROOT = ROOT

P = 7
M = 127
SUBSET_SIZE = 1_000_000
SEED = 20260417
ETAS = [0.0, 0.1, 0.3, 0.5, 0.7, 0.9]

SAVE_PLOTS = True
PLOT_EXPORT_DIR = default_output_dir(ROOT, p=P, m=M, subset_size=SUBSET_SIZE, seed=SEED)
PLOT_EXPORT_DIR


In [ ]:
# Discover matching runs from Hydra out-dir names and load their eval histories.
records = discover_modadd_runs(
    RUN_ROOT,
    p=P,
    m=M,
    subset_size=SUBSET_SIZE,
    seed=SEED,
    etas=ETAS,
)

if not records:
    raise RuntimeError(
        "No matching ModAdd runs were found. Check RUN_ROOT / P / M / SUBSET_SIZE / SEED / ETAS against your out-dir names."
    )

runs_df, run_data = build_runs_df(records)
runs_df


In [ ]:
# Quick coverage summary so we can confirm each eta has the methods we expect.
coverage_df = (
    pd.concat(run_data.values(), ignore_index=True)
    .groupby(["method", "eta"], as_index=False)
    .agg(
        n_points=("iter", "size"),
        min_iter=("iter", "min"),
        max_iter=("iter", "max"),
        final_clean_full_exact=("val/clean_full_exact", "last"),
        final_clean_final_exact=("val/clean_final_exact", "last"),
    )
    .sort_values(["eta", "method"])
    .reset_index(drop=True)
)
coverage_df


In [ ]:
# Save one per-eta figure for clean_full_exact and also show it in the notebook.
plot_per_eta(
    run_data,
    runs_df,
    metric="val/clean_full_exact",
    out_dir=PLOT_EXPORT_DIR if SAVE_PLOTS else None,
    show=True,
)


In [ ]:
# Save one per-eta figure for clean_final_exact and also show it in the notebook.
plot_per_eta(
    run_data,
    runs_df,
    metric="val/clean_final_exact",
    out_dir=PLOT_EXPORT_DIR if SAVE_PLOTS else None,
    show=True,
)
